In [ ]:
"""
Notebook: mapeamento_hipoteses.py
Projeto: Brasil Público — Escala 6x1
Etapa: 03_modelagem

Objetivo: organizar os resultados como instrumento de mapeamento
          de hipóteses — não como estimativa pontual.

Estrutura:
  1. Matriz principal: ΔPIB × faixa × produtividade (Formulação C)
  2. Heatmap com overlay de plausibilidade
  3. Normalização setorial: impacto absoluto vs intensidade relativa
  4. Posicionamento da CNI no espaço de hipóteses
  5. Faixas de resultado: plausível / estresse / estrutural / compensação

Restrições metodológicas aplicadas:
  - Nenhum cenário é destacado como "principal"
  - CNI posicionada sem inferência causal
  - Faixa de incerteza interna ao modelo — não banda completa do problema
  - Separação explícita entre resultado do modelo e interpretação econômica
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path

BASE      = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN  = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN    = BASE / "data/processed/pib_setorial_2012_2023.csv"
DIR_FIGS  = BASE / "outputs/figures"
DIR_TABS  = BASE / "outputs/tables"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

# Formulação C: média ponderada uniforme dentro de 41–44h
horas_faixa = np.array([41, 42, 43, 44])
DELTA_H_C   = ((40 - horas_faixa) / horas_faixa).mean()

# Formulação B (limite mecânico): apenas para bound superior
DELTA_H_B   = (40 - 44) / 44

correspondencia = [
    ("Agropecuária",          "agropecuaria",          True,  "direta"),
    ("Indústria geral",       "ind_transformacao",     True,  "parcial"),
    ("Construção",            "construcao",            True,  "direta"),
    ("Comércio e rep.",       "comercio",              True,  "direta"),
    ("Transp. e armaz.",      "transporte",            True,  "direta"),
    ("Inf. e serv. prof.",    "info_comunicacao",      True,  "parcial"),
    ("Adm. públ. e educação", "adm_publica_saude_educ",True, "ampla"),
    ("Alojamento e alim.",    "outros_servicos",       False, "muito_imperfeita"),
    ("Saúde",                 "adm_publica_saude_educ",False, "nao_separavel"),
    ("Serv. domésticos",      "outros_servicos",       False, "muito_imperfeita"),
]

# Parcelas afetadas por faixa
df_horas["pct_40_44_incl"]  = df_horas["pct_40h_exatas"] + df_horas["pct_41_44h"]
df_horas["pct_40_48_incl"]  = (df_horas["pct_40h_exatas"] + df_horas["pct_41_44h"]
                                + df_horas["pct_acima_44h"] * 0.44)
df_horas["pct_todos"]       = 100.0

# Fator de produtividade diferenciado por intensidade de jornada
def fator_prod(h):
    if h > 41:   return 1.2
    elif h > 39: return 1.0
    else:        return 0.8

print(f"ΔH Formulação C: {DELTA_H_C*100:.4f}%")
print(f"ΔH Formulação B: {DELTA_H_B*100:.4f}% (limite mecânico)")
print(f"PIB total 2023:  R$ {pib_total_rs/1e12:.2f} tri")

In [ ]:
def calcular_delta_pib(faixa_col, delta_h, ganho_p_global, incluir_excluidos=False):
    """
    Calcula ΔPIB dado:
      faixa_col: coluna de % afetados no df_horas
      delta_h: intensidade da redução de horas
      ganho_p_global: ganho de produtividade global (antes do fator setorial)
      incluir_excluidos: se True, inclui setores com proxy imperfeita
    """
    delta_vab_total = 0
    detalhes = []

    for setor, col, incluir, qualidade in correspondencia:
        if not incluir and not incluir_excluidos:
            continue
        if setor not in df_horas.index:
            continue

        vab_rs   = pib_2023[col] * 1e6
        h_media  = df_horas.loc[setor, "media"]
        parcela  = df_horas.loc[setor, faixa_col] / 100.0
        fator    = fator_prod(h_media)
        ganho_p  = ganho_p_global * fator

        delta_vab = vab_rs * parcela * (delta_h + ganho_p)
        delta_vab_total += delta_vab

        detalhes.append({
            "setor": setor, "incluir": incluir, "qualidade": qualidade,
            "vab_r$_bi": round(vab_rs/1e9, 2),
            "parcela": round(parcela, 4),
            "delta_h": round(delta_h*100, 4),
            "ganho_p_efetivo": round(ganho_p*100, 4),
            "delta_vab_r$_bi": round(delta_vab/1e9, 3),
            "intensidade_pct": round(parcela * (delta_h + ganho_p) * 100, 4),
        })

    return delta_vab_total / pib_total_rs * 100, pd.DataFrame(detalhes)

In [ ]:
FAIXAS = {
    "41–44h":        ("pct_41_44h",       DELTA_H_C, "baseline_consistente"),
    "40–44h":        ("pct_40_44_incl",   DELTA_H_C, "cenario_estresse"),
    "40–48h":        ("pct_40_48_incl",   DELTA_H_C, "cenario_estresse"),
    "todos_ocupados":("pct_todos",        DELTA_H_C, "limite_estrutural"),
}

GANHOS = {"0%": 0.0, "+3%": 0.03, "+6%": 0.06, "+9%": 0.09}

TARGET_CNI = -0.700

resultados = []

for nome_faixa, (col_pct, dh, cat_faixa) in FAIXAS.items():
    for nome_ganho, ganho in GANHOS.items():

        # Setores incluídos (limite inferior)
        dpib_incl, _ = calcular_delta_pib(col_pct, dh, ganho, False)
        # Todos os setores (limite superior)
        dpib_total, _ = calcular_delta_pib(col_pct, dh, ganho, True)

        # Classificação final
        if dpib_incl >= 0:
            categoria = "fronteira_compensacao"
        elif cat_faixa == "limite_estrutural":
            categoria = "limite_estrutural"
        elif cat_faixa == "cenario_estresse":
            categoria = "cenario_estresse"
        else:
            categoria = "baseline_consistente"

        dist_cni = abs(dpib_incl - TARGET_CNI)

        resultados.append({
            "faixa":          nome_faixa,
            "ganho_p":        nome_ganho,
            "delta_pib_incl": round(dpib_incl, 4),
            "delta_pib_total":round(dpib_total, 4),
            "categoria":      categoria,
            "dist_cni":       round(dist_cni, 4),
        })

df_mat = pd.DataFrame(resultados)
df_mat.to_csv(DIR_TABS / "matriz_hipoteses_completa.csv", index=False)

# Pivot para visualização
pivot = df_mat.pivot_table(
    index="faixa", columns="ganho_p",
    values="delta_pib_incl"
).reindex(["41–44h", "40–44h", "40–48h", "todos_ocupados"])[["0%","+3%","+6%","+9%"]]

print("── MATRIZ PRINCIPAL: ΔPIB (%) por faixa × ganho de produtividade ──")
print("   (setores incluídos, Formulação C, sem Formulação B)")
print()
print(pivot.round(4).to_string())
print()

# Faixas de resultado
bl = df_mat[df_mat["categoria"] == "baseline_consistente"]["delta_pib_incl"]
es = df_mat[df_mat["categoria"] == "cenario_estresse"]["delta_pib_incl"]
fc = df_mat[df_mat["categoria"] == "fronteira_compensacao"]["delta_pib_incl"]

print("── FAIXAS DE RESULTADO ──────────────────────────────────────────────")
print(f"  Baseline consistente:     [{bl.min():.4f}%, {bl.max():.4f}%]")
print(f"  Cenário de estresse:      [{es.min():.4f}%, {es.max():.4f}%]")
print(f"  Fronteira de compensação: {len(fc)} combinações com ΔPIB ≥ 0")
print()
print("── POSICIONAMENTO DA CNI (-0,700%) ──────────────────────────────────")
prox = df_mat.nsmallest(3, "dist_cni")[
    ["faixa", "ganho_p", "delta_pib_incl", "categoria", "dist_cni"]
]
print(prox.to_string(index=False))
print()
print("Nota: CNI posicionada por proximidade numérica.")
print("Não implica equivalência metodológica nem validação cruzada.")

In [ ]:
print("\n── NORMALIZAÇÃO SETORIAL ────────────────────────────────────────────")
print("Cenário de referência: faixa 41–44h, ganho 0% (Formulação C)")

_, df_det = calcular_delta_pib("pct_41_44h", DELTA_H_C, 0.0, True)

df_norm = df_det.copy()
df_norm["participacao_pib_pct"] = df_norm["vab_r$_bi"] / (pib_total_rs/1e9) * 100
df_norm["impacto_norm"] = (
    df_norm["delta_vab_r$_bi"] / df_norm["vab_r$_bi"] * 100
)  # impacto como % do VAB próprio

print("\n Setor | Part. PIB | Impacto R$bi | Intensidade (%VAB próprio)")
print("-" * 75)
for _, r in df_norm.sort_values("delta_vab_r$_bi").iterrows():
    inc = "✓" if r["incluir"] else "✗"
    print(f" {inc} {r['setor']:<28} {r['participacao_pib_pct']:>6.2f}%    "
          f"R$ {r['delta_vab_r$_bi']:>7.3f} bi    {r['impacto_norm']:>8.3f}%")

print()
print("✓ = incluído na simulação  ✗ = excluído (proxy imperfeita)")
print("Intensidade: impacto / VAB do próprio setor — separa 'setor grande' de 'setor afetado'")

df_norm.to_csv(DIR_TABS / "normalizacao_setorial.csv", index=False)

In [ ]:
pivot_cat = df_mat.pivot_table(
    index="faixa", columns="ganho_p", values="categoria", aggfunc="first"
).reindex(["41–44h", "40–44h", "40–48h", "todos_ocupados"])[["0%","+3%","+6%","+9%"]]

data  = pivot.values.astype(float)
vmax  = max(abs(data.min()), abs(data.max()))
cmap  = plt.colormaps["RdYlGn"]
norm  = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")

ax.set_xticks(range(4))
ax.set_xticklabels([f"Ganho\n{c}" for c in pivot.columns], fontsize=10)
ax.set_yticks(range(4))
ax.set_yticklabels(pivot.index, fontsize=10)

simbolo = {
    "baseline_consistente":  "●",
    "cenario_estresse":      "▲",
    "limite_estrutural":     "■",
    "fronteira_compensacao": "★",
}

CNI_LABEL_ADDED = False

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        faixa_n = pivot.index[i]
        ganho_n = pivot.columns[j]
        cat     = pivot_cat.loc[faixa_n, ganho_n]
        sim     = simbolo.get(cat, "")
        cor_txt = "white" if abs(val) > vmax * 0.45 else "black"

        # Marcar CNI
        dist = abs(val - TARGET_CNI)
        cni_mark = f"\n≈CNI" if dist <= 0.08 else ""

        ax.text(j, i, f"{val:.3f}%\n{sim}{cni_mark}",
                ha="center", va="center", fontsize=8.5,
                color=cor_txt, fontweight="bold")

        # Borda dourada na fronteira de compensação
        if val >= 0:
            ax.add_patch(plt.Rectangle(
                (j-0.5, i-0.5), 1, 1,
                fill=False, edgecolor="gold", linewidth=2.5
            ))

        # Borda tracejada nos próximos da CNI
        if 0 < dist <= 0.08:
            ax.add_patch(plt.Rectangle(
                (j-0.5, i-0.5), 1, 1,
                fill=False, edgecolor="navy", linewidth=2, linestyle="--"
            ))

plt.colorbar(im, ax=ax, label="ΔPIB estimado (%)", shrink=0.8)

ax.set_title(
    "Espaço de resultados: redução de jornada (44h→40h) — Brasil 2023\n"
    "Formulação C · Produtividade diferenciada por setor · Setores incluídos",
    fontsize=12, loc="left"
)

legenda_items = [
    mpatches.Patch(color="white", label="● Baseline consistente"),
    mpatches.Patch(color="white", label="▲ Cenário de estresse"),
    mpatches.Patch(color="white", label="■ Limite estrutural"),
    mpatches.Patch(color="white", label="★ Fronteira compensação (ΔPIB≥0)"),
    mpatches.Patch(color="navy",  label="── Próximo à CNI (±0,08pp)"),
    mpatches.Patch(color="gold",  label="── Fronteira compensação total"),
]
ax.legend(handles=legenda_items, loc="upper left", bbox_to_anchor=(1.15, 1),
          frameon=False, fontsize=8.5)

fig.text(
    0.01, -0.03,
    f"Faixa plausível: [{bl.min():.3f}%, {bl.max():.3f}%]  |  "
    f"CNI: -0,700% (sem equivalência metodológica)  |  "
    "Faixa de incerteza interna ao modelo — não banda completa do problema econômico.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "12_mapeamento_hipoteses.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 12 salvo.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_plot = df_norm[df_norm["incluir"] == True].copy()
df_plot = df_plot.sort_values("impacto_norm", ascending=True)

qualidade_cores = {"direta": "#2C6FBF", "parcial": "#F5A623", "ampla": "#E84545"}
cores = [qualidade_cores.get(q, "#888") for q in df_plot["qualidade"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

bars1 = ax1.barh(df_plot["setor"], df_plot["delta_vab_r$_bi"], color=cores, height=0.6)
ax1.axvline(0, color="black", linewidth=0.8)

for bar, (_, r) in zip(bars1, df_plot.iterrows()):
    val = r["delta_vab_r$_bi"]
    if val < -15:
        x_pos = val + 0.5
        align = 'left'
        color_text = 'white'
    else:
        x_pos = val - 0.5
        align = 'right'
        color_text = 'black'
        
    ax1.text(x_pos, bar.get_y() + bar.get_height()/2,
             f"R$ {val:.2f} bi\n({r['participacao_pib_pct']:.1f}% do PIB)",
             va="center", ha=align, fontsize=7.5, 
             color=color_text, linespacing=1.1)

ax1.set_xlabel("Contribuição para ΔPIB (R$ bilhões)", fontsize=10)
ax1.set_title("Impacto absoluto\n(tamanho do setor influencia)", fontsize=11, loc="left")
ax1.set_xlim(df_plot["delta_vab_r$_bi"].min() * 1.25, 0.5) 

bars2 = ax2.barh(df_plot["setor"], df_plot["impacto_norm"], color=cores, height=0.6)
ax2.axvline(0, color="black", linewidth=0.8)

for bar, (_, r) in zip(bars2, df_plot.iterrows()):
    val = r["impacto_norm"]
    if val < -1.5:
        x_pos = val + 0.05
        align = 'left'
        color_text = 'white'
    else:
        x_pos = val - 0.05
        align = 'right'
        color_text = 'black'

    ax2.text(x_pos, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}%\n(pct_41-44h: {r['parcela']*100:.1f}%)",
             va="center", ha=align, fontsize=7.5, 
             color=color_text, linespacing=1.1)

ax2.set_xlabel("Impacto / VAB do próprio setor (%)", fontsize=10)
ax2.set_title("Intensidade relativa\n(independente do tamanho)", fontsize=11, loc="left")
ax2.set_xlim(df_plot["impacto_norm"].min() * 1.2, 0.05)

legenda = [
    mpatches.Patch(color="#2C6FBF", label="Proxy direta"),
    mpatches.Patch(color="#F5A623", label="Proxy parcial"),
    mpatches.Patch(color="#E84545", label="Proxy ampla"),
]
fig.legend(handles=legenda, loc="lower center", ncol=3,
           frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.05))

fig.suptitle(
    "Normalização setorial: impacto absoluto vs intensidade relativa — Brasil 2023\n"
    "Cenário: faixa 41–44h, ganho 0%, Formulação C · PNAD Contínua + CNT/IBGE",
    fontsize=12, y=1.01
)

fig.text(
    0.01, -0.08,
    "Intensidade relativa = ΔVAB / VAB do setor. "
    "Separa 'setor grande com impacto grande' de 'setor estruturalmente afetado'. "
    "Limitação: proxy setorial imperfeita em setores marcados com cor laranja/vermelho.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "13_normalizacao_setorial.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 13 salvo com correção de alinhamento.")

In [ ]:
# Derivar todos os valores diretamente de df_mat
bl       = df_mat[df_mat["categoria"] == "baseline_consistente"]["delta_pib_incl"]
es       = df_mat[df_mat["categoria"] == "cenario_estresse"]["delta_pib_incl"]
ls       = df_mat[df_mat["categoria"] == "limite_estrutural"]["delta_pib_incl"]
fc       = df_mat[df_mat["categoria"] == "fronteira_compensacao"]["delta_pib_incl"]
all_vals = df_mat["delta_pib_incl"]

# Combinação mais próxima da CNI (dentro do baseline)
mais_prox = df_mat.nsmallest(1, "dist_cni").iloc[0]
dist_cni  = mais_prox["dist_cni"]

# Fronteira de compensação: menor ganho que inverte sinal no baseline
fc_baseline = df_mat[
    (df_mat["faixa"] == "41–44h") &
    (df_mat["delta_pib_incl"] >= 0)
]["ganho_p"].min() if len(df_mat[
    (df_mat["faixa"] == "41–44h") &
    (df_mat["delta_pib_incl"] >= 0)
]) > 0 else "não atingida"

print(f"""
══ SUMÁRIO DO ESPAÇO DE HIPÓTESES ══════════════════════════════════════

Este instrumento não produz uma estimativa única do impacto da redução
de jornada. Produz um mapeamento condicionado a hipóteses explícitas.

AMPLITUDE TOTAL DO ESPAÇO:
  ΔPIB varia de {all_vals.min():.3f}% a {all_vals.max():.3f}% dependendo das hipóteses.
  Isso não reflete incerteza sobre um fato — reflete que hipóteses
  diferentes correspondem a mecanismos econômicos diferentes.

FAIXAS POR CATEGORIA (setores incluídos, Formulação C):
  Baseline consistente (faixa 41–44h):  [{bl.min():.3f}%, {bl.max():.3f}%]
  Cenário de estresse (faixas 40–44h / 40–48h): [{es.min():.3f}%, {es.max():.3f}%]
  Limite estrutural (todos os ocupados): [{ls.min():.3f}%, {ls.max():.3f}%]
  Fronteira de compensação (ΔPIB ≥ 0):  {len(fc)} combinações

FRONTEIRA DE COMPENSAÇÃO NO BASELINE:
  Menor ganho de produtividade que inverte o sinal (faixa 41–44h): {fc_baseline}
  Nota: propriedade do modelo — depende de produtividade exógena.
  Não incorpora dinâmica de mercado de trabalho.

POSICIONAMENTO DA CNI (-0,700%):
  Combinação mais próxima: faixa {mais_prox['faixa']}, ganho {mais_prox['ganho_p']}
  Valor: {mais_prox['delta_pib_incl']:.4f}%  |  Distância: {dist_cni:.4f} pp
  Categoria: {mais_prox['categoria']}

  Isso indica proximidade de ordem de grandeza.
  NÃO indica equivalência metodológica nem validação cruzada.
  Os modelos partem de estruturas diferentes e hipóteses não comparáveis.

HETEROGENEIDADE SETORIAL (cenário baseline 0% produtividade):
  Impacto NÃO é homogêneo entre setores.
  Setores com maior intensidade relativa (impacto / VAB próprio):
    Comércio e rep.: intensidade relevante com {df_norm[df_norm['setor']=='Comércio e rep.']['impacto_norm'].values[0]:.3f}% do VAB próprio
    Indústria geral: {df_norm[df_norm['setor']=='Indústria geral']['impacto_norm'].values[0]:.3f}% do VAB próprio
  Setor menos intenso (entre os incluídos):
    Adm. públ. e educação: {df_norm[df_norm['setor']=='Adm. públ. e educação']['impacto_norm'].values[0]:.3f}% do VAB próprio

LIMITAÇÕES QUE ESTA ANÁLISE NÃO RESOLVE:
  - Produtividade exógena e uniforme dentro do setor
  - Correspondência setorial PNAD × CNT imperfeita
  - Efeitos de segunda ordem ausentes (demanda, preços, competitividade)
  - Setor informal (~39% dos ocupados) captado parcialmente
  - Formulação C assume distribuição uniforme dentro de 41–44h
  Ver DECISIONS.md para documentação completa.
""")
